## Análisis de sentimientos de Deep Learning

### Bloque 1: Preparación de Datos

In [5]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
# Datos de ejemplo: 0 para Negativo, 1 para Positivo
frases = [
    "Me encanta este producto, es increíble", "Excelente calidad y muy rápido",
    "Es lo mejor que he comprado nunca", "Muy feliz con mi compra",
    "No me gusta nada, es un desastre", "Pésimo servicio y llegó roto",
    "Una pérdida de dinero total", "No lo recomiendo para nada, muy malo"
]

etiquetas = [1, 1, 1, 1, 0, 0, 0, 0]  # 1=Pos, 0=Neg

# Convertir a arrays de numpy
etiquetas = np.array(etiquetas)

### Bloque 2: Tokeninzación y Padding

In [6]:
# Define cuántas palabras distintas va a recordar el modelo. Aquí le decimos: "Solo quédate con las 500 palabras más frecuentes".
# Si el texto tiene palabras muy raras, las ignorará.
vocab_size = 500

# Define la longitud máxima de las frases, en este caso 10 palabras
max_length = 10

# Indica que si una frase excede las 10 palabras se cortará el final
trunc_type = 'post'

# Indica que si una frase no llega a 10 palabras se completará con 0 al final
padding_type = 'post'

# Significa Out of Vocabulary, si una palabra introducida no ha sido contemplada en el entrenamiento simplemente se sustituye por ese token especial
oov_tok = "<OOV>"
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)

# Aprendizaje: lee todas las frases y crea un diccionario interno.
tokenizer.fit_on_texts(frases)

# Transformación: convierte las listas de palabras en listas de números usando el diccionario anterior. “Me encanta el producto" se convierte en [1, 2, 4, 3].
sequences = tokenizer.texts_to_sequences(frases)

# Homogeneización: aplicamos el max_length definido anteriormente para que todas las frases tengan el mismo tamaño
padded = pad_sequences(
    sequences,
    maxlen=max_length,
    padding=padding_type,
    truncating=trunc_type,
)

### Bloque 3: El modelo con LSTM (Memoria a largo plazo)

In [7]:
model = tf.keras.Sequential([
    # Capa de Embedding (crea el mapa de significados)
    # En esta capa el modelo aprende a colocar las palabras cerca o
    # lejos vectorialmente hablando en función de su significado
    tf.keras.layers.Embedding(
        vocab_size,
        16,
        input_length=max_length,
    ),

    # Capa LSTM: esta es la que "entiende" el orden y contexto
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),

    # Capa de 16 neuronas para decidir el resultado
    tf.keras.layers.Dense(16, activation='relu'),

    # Capa de salida: 1 sola neurona con Sigmoid (devuelve un valor
    # entre 0 y 1)
    tf.keras.layers.Dense(1, activation='sigmoid')
])
# loss='binary_crossentropy': es la fórmula matemática para medir cuánto se
# equivoca el modelo. Como solo comparamos dos cosas (0 o 1), esta es la
# mejor fórmula.
# optimizer='adam': es el "entrenador". Es el algoritmo que decide cómo
# ajustar los pesos de las neuronas para que el modelo aprenda de sus
# errores.
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy'],
)
model.summary()

/home/ciabd10/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
W0000 00:00:1775571060.548441   11700 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Bloque 4: Entrenamiento

In [8]:
# Entrenamos (con pocos datos necesitamos muchas épocas)
model.fit(padded, etiquetas, epochs=30, verbose=0)
print("Modelo entrenado.")

Modelo entrenado.


### Bloque 5: Probando el sentimiento con nuevas frases

In [9]:
def predecir_sentimiento(nueva_frase):
    secuencia = tokenizer.texts_to_sequences([nueva_frase])
    relleno = pad_sequences(
        secuencia,
        maxlen=max_length,
        padding=padding_type,
    )

    prediccion = model.predict(relleno, verbose=0)[0][0]

    if prediccion > 0.5:
        return f" POSITIVO ({prediccion:.2f})"
    else:
        return f" NEGATIVO ({prediccion:.2f})"
# Pruebas
print(predecir_sentimiento("Es un producto fantástico"))
print(predecir_sentimiento("Menuda basura de compra"))
print(predecir_sentimiento("No es tan bueno como pensaba"))

 NEGATIVO (0.49)
 NEGATIVO (0.47)
 NEGATIVO (0.47)
